<a href="https://colab.research.google.com/github/mafedcp65-netizen/Trabajo-de-grado/blob/main/Modelo_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo Transformers

## Entorno

In [1]:
import sys
import subprocess
import importlib.metadata as im

REQUERIDOS = {
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "scipy": "1.13.1",
    "scikit-learn": "1.5.2",
    "openpyxl": "3.1.5",
    "transformers": "4.46.3",
    "datasets": "3.1.0",
    "accelerate": "1.1.1",
    "evaluate": "0.4.3",
    "python-docx": "1.1.2",
}

def version_instalada(pkg):
    try:
        return im.version(pkg)
    except Exception:
        return None

def instalar_faltantes():
    pendientes = []
    for pkg, ver in REQUERIDOS.items():
        instalada = version_instalada(pkg)
        if instalada != ver:
            pendientes.append(f"{pkg}=={ver}")
    if pendientes:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pendientes]
        subprocess.check_call(cmd)
        print("Se instalaron/actualizaron paquetes.")
    else:
        print("Las versiones requeridas ya están instaladas.")


instalar_faltantes()

Se instalaron/actualizaron paquetes.


## Carga del archivo

In [1]:
try:
    from google.colab import files
    uploaded = files.upload()
    ARCHIVO = list(uploaded.keys())[0]
    print(f"Archivo cargado: {ARCHIVO}")
except Exception:
    ARCHIVO = "base etiquetada limpia.xlsx"
    print(f"Entorno local/Jupyter detectado. Se usará: {ARCHIVO}")

Saving base etiquetada limpia.xlsx to base etiquetada limpia.xlsx
Archivo cargado: base etiquetada limpia.xlsx


## Imports y configuración

In [2]:
import os
import re
import json
import random
import inspect
import unicodedata
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ARCHIVO_EXCEL = ARCHIVO
HOJA = "Base_estandarizada"
COLUMNA_ID = "ID_PSEUDO"
COLUMNAS_TEXTO = [
    "Diagnóstico A",
    "Diagnóstico B",
    "Diagnóstico C",
    "Diagnóstico D",
    "Otros estados patológicos",
    "Otros estados patológicos 2",
]
COLUMNA_ETIQUETA = "Mención de cáncer"

MODELO_HF = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
MAX_LENGTH = 128
TEST_SIZE = 0.20
VAL_SIZE_RELATIVO = 0.20
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16 if torch.cuda.is_available() else 4
EVAL_BATCH_SIZE = 32 if torch.cuda.is_available() else 8
EARLY_STOPPING_PATIENCE = 2

UMBRAL_BASE_0 = 0.50
UMBRAL_BASE_1 = 0.40
UMBRAL_BASE_2 = 0.30
UMBRAL_BASE_ANY_CANCER = 0.35
MARGEN_BASE_1_SOBRE_2 = 0.05

CARPETA_SALIDA = Path("resultados_roberta_hf_multiclase_umbralizado")
CARPETA_MODELO = CARPETA_SALIDA / "modelo_final"
RUTA_MODELO_INFERENCIA = CARPETA_MODELO
RUTA_UMBRALES_INFERENCIA = CARPETA_SALIDA / "mejores_umbrales.json"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
CARPETA_MODELO.mkdir(parents=True, exist_ok=True)

INVALIDOS_TEXTO = {"", "0", "....", "NO PRESENTO NINGUNO", "SE DESCONOCE"}
LABELS = np.array([0, 1, 2], dtype=int)

print("Usando GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Usando GPU: True
GPU: Tesla T4


## Funciones auxiliares

In [3]:
def guardar_json(ruta, data):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def normalize_text(s):
    s = str(s).lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", s).strip()


PATRONES_SOSPECHA = [
    "sospecha",
    "probable",
    "posible",
    "masa ",
    "masa de",
    "masa pulmonar",
    "masa hepatica",
    "comportamiento incierto",
    "comportamiento desconocido",
    "incierto",
    "desconocido",
    "a descartar",
    "por descartar",
    "tumoracion",
    "lesion ",
    "lesion hepatica",
]
PATRONES_SOSPECHA_NORMALIZADOS = [normalize_text(p) for p in PATRONES_SOSPECHA]

def tiene_patron_sospecha_normalizado(texto_normalizado):
    return int(any(p in texto_normalizado for p in PATRONES_SOSPECHA_NORMALIZADOS))

def softmax_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 1:
        x = x.reshape(1, -1)
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=1, keepdims=True)

def metricas_multiclase(y_true, y_pred, labels=LABELS):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    cm = confusion_matrix(y_true, y_pred, labels=list(labels)).astype(np.float64)
    total = cm.sum()
    soporte = cm.sum(axis=1)
    predichos = cm.sum(axis=0)
    tp = np.diag(cm)
    fp = predichos - tp
    fn = soporte - tp
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)
    accuracy = float(tp.sum() / total) if total else 0.0
    metricas = {
        "accuracy": accuracy,
        "precision_macro": float(precision.mean()),
        "recall_macro": float(recall.mean()),
        "f1_macro": float(f1.mean()),
    }
    pesos = soporte / soporte.sum() if soporte.sum() else np.zeros_like(soporte)
    metricas["precision_weighted"] = float((precision * pesos).sum()) if soporte.sum() else 0.0
    metricas["recall_weighted"] = float((recall * pesos).sum()) if soporte.sum() else 0.0
    metricas["f1_weighted"] = float((f1 * pesos).sum()) if soporte.sum() else 0.0
    for i, clase in enumerate(labels):
        metricas[f"precision_clase_{clase}"] = float(precision[i])
        metricas[f"recall_clase_{clase}"] = float(recall[i])
        metricas[f"f1_clase_{clase}"] = float(f1[i])
        metricas[f"soporte_clase_{clase}"] = int(soporte[i])
    return metricas

def metricas_any_cancer(y_true, y_pred):
    y_true_bin = np.isin(np.asarray(y_true, dtype=int), [1, 2]).astype(int)
    y_pred_bin = np.isin(np.asarray(y_pred, dtype=int), [1, 2]).astype(int)
    tp = int(((y_true_bin == 1) & (y_pred_bin == 1)).sum())
    fp = int(((y_true_bin == 0) & (y_pred_bin == 1)).sum())
    fn = int(((y_true_bin == 1) & (y_pred_bin == 0)).sum())
    tn = int(((y_true_bin == 0) & (y_pred_bin == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
    }


def classification_report_df(y_true, y_pred, labels=LABELS):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    cm = confusion_matrix(y_true, y_pred, labels=list(labels)).astype(np.float64)
    soporte = cm.sum(axis=1)
    predichos = cm.sum(axis=0)
    tp = np.diag(cm)
    fp = predichos - tp
    fn = soporte - tp
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)
    return pd.DataFrame({
        "clase": labels,
        "precision": precision,
        "sensibilidad": recall,
        "f1": f1,
        "soporte": soporte.astype(int),
    })


def predecir_con_umbrales(probabilidades, cues_sospecha, umbrales):
    probs = np.asarray(probabilidades, dtype=np.float64)
    cues = np.asarray(cues_sospecha, dtype=bool)
    p0 = probs[:, 0]
    p1 = probs[:, 1]
    p2 = probs[:, 2]
    cond1 = (p1 >= umbrales["t1"]) & (p1 >= p2 + umbrales["margin12"])
    cond2 = (p2 >= umbrales["t2"]) | (cues & ((p1 + p2) >= umbrales["t_any"]) & (p1 < umbrales["t1"]))
    cond0 = p0 >= umbrales["t0"]
    pred_argmax = np.argmax(probs, axis=1).astype(int)
    preds = pred_argmax.copy()
    reglas = np.full(len(preds), "argmax_respaldo", dtype=object)
    idx0 = cond0 & ~cond1 & ~cond2
    preds[idx0] = 0
    reglas[idx0] = "0_por_umbral_negativo"
    idx2 = cond2 & ~cond1
    preds[idx2] = 2
    reglas[idx2] = "2_por_umbral_sospecha"
    preds[cond1] = 1
    reglas[cond1] = "1_por_umbral_claro"
    return preds, reglas.tolist()


def buscar_mejores_umbrales(probs_val, y_val, cues_val):
    mejores = None
    mejores_metricas = None
    mejor_score = -1.0
    grid_t0 = [0.50, 0.55, 0.60]
    grid_t1 = [0.40, 0.45, 0.50, 0.55]
    grid_t2 = [0.12, 0.15, 0.18, 0.22, 0.25, 0.30]
    grid_tany = [0.35, 0.40, 0.45, 0.50]
    grid_margin = [0.05, 0.10, 0.15]
    for t0, t1, t2, t_any, margin12 in product(grid_t0, grid_t1, grid_t2, grid_tany, grid_margin):
        umbrales = {"t0": t0, "t1": t1, "t2": t2, "t_any": t_any, "margin12": margin12}
        pred_val, _ = predecir_con_umbrales(probs_val, cues_val, umbrales)
        mets = metricas_multiclase(y_val, pred_val, labels=LABELS)
        score = 0.55 * mets.get("recall_clase_2", 0.0) + 0.25 * mets.get("recall_macro", 0.0) + 0.20 * mets.get("f1_macro", 0.0)
        if score > mejor_score:
            mejor_score = score
            mejores = umbrales
            mejores_metricas = mets
    return mejores, mejores_metricas, mejor_score


def balanced_class_weights(y):
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=len(LABELS)).astype(np.float64)
    weights = np.zeros(len(LABELS), dtype=np.float64)
    present = counts > 0
    n_present = int(present.sum()) if present.sum() else len(LABELS)
    weights[present] = len(y) / (n_present * counts[present])
    if present.any():
        weights[present] = np.sqrt(weights[present])
        weights[present] = weights[present] / weights[present].mean()
    return weights

def preparar_dataframe(df_base):
    df = df_base.copy()
    for col in COLUMNAS_TEXTO:
        serie = df[col].fillna("").astype(str).str.strip()
        invalida = serie.str.lower().isin({"nan", "none", "null"}) | serie.isin(INVALIDOS_TEXTO)
        df[col] = serie.mask(invalida, "")
    partes = []
    for col in COLUMNAS_TEXTO:
        serie = df[col]
        partes.append(np.where(serie.ne(""), f"{col}: " + serie, ""))
    texto_array = partes[0]
    for extra in partes[1:]:
        texto_array = np.where((texto_array != "") & (extra != ""), texto_array + " [SEP] " + extra, np.where(texto_array != "", texto_array, extra))
    df["texto_modelo"] = pd.Series(texto_array, index=df.index).astype(str).str.strip()
    df = df[df["texto_modelo"].str.len() > 0].copy()
    df["texto_normalizado"] = df["texto_modelo"].map(normalize_text)
    df["tiene_cue_sospecha"] = df["texto_normalizado"].map(tiene_patron_sospecha_normalizado).astype(int)
    etiqueta_disponible = COLUMNA_ETIQUETA in df.columns
    if etiqueta_disponible:
        df[COLUMNA_ETIQUETA] = df[COLUMNA_ETIQUETA].fillna("").astype(str).str.strip()
        df["etiqueta_real"] = pd.to_numeric(df[COLUMNA_ETIQUETA], errors="coerce")
        df.loc[~df["etiqueta_real"].isin([0, 1, 2]), "etiqueta_real"] = np.nan
    else:
        df["etiqueta_real"] = np.nan
    return df


def tokenizar_dataset_dict(dataset_dict, tokenizer):
    def tokenize_batch(batch):
        return tokenizer(batch["texto_modelo"], truncation=True, max_length=MAX_LENGTH, padding=False)
    tokenized = dataset_dict.map(tokenize_batch, batched=True)
    for split in list(tokenized.keys()):
        tokenized[split] = tokenized[split].rename_column("etiqueta_real", "labels")
        cols_to_keep = ["input_ids", "attention_mask", "labels"]
        remove_cols = [c for c in tokenized[split].column_names if c not in cols_to_keep]
        tokenized[split] = tokenized[split].remove_columns(remove_cols)
        tokenized[split].set_format(type="torch")
    return tokenized


def carpeta_modelo_hf_valida(ruta):
    ruta = Path(ruta)
    archivos_minimos = ["config.json"]
    return ruta.exists() and ruta.is_dir() and all((ruta / nombre).exists() for nombre in archivos_minimos)


def predecir_dataframe_modelo(df_inferencia, model, tokenizer, batch_size):
    textos = df_inferencia["texto_modelo"].astype(str).tolist()
    if len(textos) == 0:
        return np.empty((0, 3)), np.empty((0, 3)), np.array([], dtype=int)
    device = next(model.parameters()).device
    model.eval()
    logits_list = []
    for i in range(0, len(textos), batch_size):
        batch_textos = textos[i:i + batch_size]
        enc = tokenizer(batch_textos, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits.detach().cpu().numpy()
        logits_list.append(logits)
    logits = np.vstack(logits_list)
    probs = softmax_numpy(logits)
    pred_argmax = np.argmax(logits, axis=1).astype(int)
    return logits, probs, pred_argmax


def resumen_split(nombre, data):
    salida = {"nombre": nombre, "n": int(len(data))}
    if "etiqueta_real" in data.columns:
        conteo = data["etiqueta_real"].value_counts(dropna=False).sort_index()
        salida["clases"] = {str(k): int(v) for k, v in conteo.items()}
    return salida

## Lectura y preparación de la base

In [4]:
assert os.path.exists(ARCHIVO_EXCEL), f"No se encontró el archivo: {ARCHIVO_EXCEL}"
xls = pd.ExcelFile(ARCHIVO_EXCEL)
if HOJA not in xls.sheet_names:
    HOJA = xls.sheet_names[0]
    print(f"La hoja solicitada no existe. Se usará la primera hoja disponible: {HOJA}")

df_raw = pd.read_excel(ARCHIVO_EXCEL, sheet_name=HOJA, dtype=str)
print("Filas iniciales:", len(df_raw))

raw_columns_normalized_map = {normalize_text(col): col for col in df_raw.columns}

rename_map = {}
all_expected_columns = [COLUMNA_ID] + COLUMNAS_TEXTO

for expected_col in all_expected_columns:
    expected_col_norm = normalize_text(expected_col)
    if expected_col_norm in raw_columns_normalized_map:
        actual_raw_col = raw_columns_normalized_map[expected_col_norm]
        if actual_raw_col != expected_col:
            rename_map[actual_raw_col] = expected_col

df_raw = df_raw.rename(columns=rename_map)

columnas_obligatorias = [COLUMNA_ID, *COLUMNAS_TEXTO]
faltantes = [c for c in columnas_obligatorias if c not in df_raw.columns]
assert not faltantes, f"Faltan columnas obligatorias: {faltantes}"

df = preparar_dataframe(df_raw)
print("Filas utilizables:", len(df))

etiquetas_validas = df["etiqueta_real"].dropna().astype(int)
conteo_etiquetas = etiquetas_validas.value_counts().sort_index()
clases_presentes = sorted(etiquetas_validas.unique().tolist())

print()
print("Registros con etiqueta válida:", int(etiquetas_validas.shape[0]))
if len(conteo_etiquetas) > 0:
    print(conteo_etiquetas)
else:
    print("No se encontraron etiquetas válidas 0/1/2.")

Filas iniciales: 4000
Filas utilizables: 4000

Registros con etiqueta válida: 3999
etiqueta_real
0    3052
1     908
2      39
Name: count, dtype: int64


## Decisión de ruta

In [5]:

minimo_por_clase_para_split = 5
hay_etiquetas_suficientes = len(clases_presentes) == 3 and len(conteo_etiquetas) == 3 and int(conteo_etiquetas.min()) >= minimo_por_clase_para_split

modo_entrenamiento = hay_etiquetas_suficientes
modo_inferencia = not modo_entrenamiento
modelo_guardado_disponible = carpeta_modelo_hf_valida(RUTA_MODELO_INFERENCIA)
puede_clasificar_total = True

if modo_entrenamiento:
    print("Se detectaron etiquetas suficientes. Se hará entrenamiento, validación, test y clasificación total.")
else:
    print("No hay etiquetas suficientes para entrenar de forma segura. Se usará el modo de inferencia.")
    print(f"Modelo esperado para inferencia: {RUTA_MODELO_INFERENCIA}")
    if modelo_guardado_disponible:
        print("Se encontró un modelo afinado guardado. Se usará para clasificar toda la base.")
    else:
        puede_clasificar_total = False
        print("No se encontró un modelo afinado guardado.")
        print("El cuaderno no se detendrá: exportará dos Excel con la estructura de salida, pero sin predicciones ni scores.")
        print("Para obtener clasificaciones con una base sin etiqueta, primero ejecuta el cuaderno con una base etiquetada y conserva la carpeta del modelo.")

Se detectaron etiquetas suficientes. Se hará entrenamiento, validación, test y clasificación total.


## Tokenizador y modelo base

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_HF, use_fast=True)
id2label = {0: "0", 1: "1", 2: "2"}
label2id = {"0": 0, "1": 1, "2": 2}
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

## Entrenamiento, calibración y evaluación

In [7]:
metricas_val_argmax = None
metricas_val_umbral = None
metricas_test_argmax = None
metricas_test_umbral = None
metricas_any_test = None
cm_test_umbral = None
reporte_test_umbral_df = None
mejores_umbrales = {
    "t0": UMBRAL_BASE_0,
    "t1": UMBRAL_BASE_1,
    "t2": UMBRAL_BASE_2,
    "t_any": UMBRAL_BASE_ANY_CANCER,
    "margin12": MARGEN_BASE_1_SOBRE_2,
}
trainer = None
model = None
splits_resumen = []
raw_class_weights = None
class_weights = None

if modo_entrenamiento:
    df_labeled = df[df["etiqueta_real"].notna()].copy()
    df_labeled["etiqueta_real"] = df_labeled["etiqueta_real"].astype(int)

    train_val_df, test_df = train_test_split(
        df_labeled,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df_labeled["etiqueta_real"],
    )
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=VAL_SIZE_RELATIVO,
        random_state=RANDOM_STATE,
        stratify=train_val_df["etiqueta_real"],
    )

    print("Train:", len(train_df))
    print("Validación:", len(val_df))
    print("Test:", len(test_df))

    raw_class_weights = balanced_class_weights(train_df["etiqueta_real"].to_numpy())
    class_weights = raw_class_weights.copy()
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
    class_to_sample_weight = {int(c): float(1.0 / n) for c, n in train_df["etiqueta_real"].value_counts().items()}
    train_sample_weights = torch.as_tensor(train_df["etiqueta_real"].map(class_to_sample_weight).to_numpy(), dtype=torch.double)

    dataset = DatasetDict({
        "train": Dataset.from_pandas(train_df[[COLUMNA_ID, "texto_modelo", "etiqueta_real", "tiene_cue_sospecha"]].reset_index(drop=True), preserve_index=False),
        "validation": Dataset.from_pandas(val_df[[COLUMNA_ID, "texto_modelo", "etiqueta_real", "tiene_cue_sospecha"]].reset_index(drop=True), preserve_index=False),
        "test": Dataset.from_pandas(test_df[[COLUMNA_ID, "texto_modelo", "etiqueta_real", "tiene_cue_sospecha"]].reset_index(drop=True), preserve_index=False),
    })
    tokenized_ds = tokenizar_dataset_dict(dataset, tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return metricas_multiclase(labels, preds, labels=LABELS)

    model = AutoModelForSequenceClassification.from_pretrained(MODELO_HF, num_labels=3, id2label=id2label, label2id=label2id)

    class WeightedSamplerTrainer(Trainer):
        def __init__(self, class_weights=None, sample_weights=None, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights
            self.sample_weights = sample_weights

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            outputs = model(input_ids=inputs.get("input_ids"), attention_mask=inputs.get("attention_mask"))
            logits = outputs.get("logits")
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

        def get_train_dataloader(self):
            sampler = WeightedRandomSampler(weights=self.sample_weights, num_samples=len(self.sample_weights), replacement=True)
            return DataLoader(
                self.train_dataset,
                batch_size=self.args.train_batch_size,
                sampler=sampler,
                collate_fn=self.data_collator,
                num_workers=self.args.dataloader_num_workers,
                pin_memory=self.args.dataloader_pin_memory,
            )

    sig = inspect.signature(TrainingArguments.__init__)
    training_kwargs = dict(
        output_dir=str(CARPETA_SALIDA / "checkpoints"),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_recall_macro",
        greater_is_better=True,
        save_total_limit=1,
        seed=RANDOM_STATE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    if "evaluation_strategy" in sig.parameters:
        training_kwargs["evaluation_strategy"] = "epoch"
    else:
        training_kwargs["eval_strategy"] = "epoch"
    if "group_by_length" in sig.parameters:
        training_kwargs["group_by_length"] = True
    if "dataloader_num_workers" in sig.parameters:
        training_kwargs["dataloader_num_workers"] = 2 if torch.cuda.is_available() else 0
    if "optim" in sig.parameters and torch.cuda.is_available():
        training_kwargs["optim"] = "adamw_torch_fused"

    training_args = TrainingArguments(**training_kwargs)

    trainer = WeightedSamplerTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights_tensor,
        sample_weights=train_sample_weights,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer.train()

    pred_val = trainer.predict(tokenized_ds["validation"])
    probs_val = softmax_numpy(pred_val.predictions)
    y_val = pred_val.label_ids
    y_pred_val_argmax = np.argmax(pred_val.predictions, axis=1)
    cues_val = val_df["tiene_cue_sospecha"].to_numpy(dtype=int)
    mejores_umbrales, metricas_val_umbral, _ = buscar_mejores_umbrales(probs_val, y_val, cues_val)
    metricas_val_argmax = metricas_multiclase(y_val, y_pred_val_argmax, labels=LABELS)

    pred_test = trainer.predict(tokenized_ds["test"])
    probs_test = softmax_numpy(pred_test.predictions)
    y_test = pred_test.label_ids
    y_pred_test_argmax = np.argmax(pred_test.predictions, axis=1)
    cues_test = test_df["tiene_cue_sospecha"].to_numpy(dtype=int)
    y_pred_test_umbral, reglas_test = predecir_con_umbrales(probs_test, cues_test, mejores_umbrales)

    metricas_test_argmax = metricas_multiclase(y_test, y_pred_test_argmax, labels=LABELS)
    metricas_test_umbral = metricas_multiclase(y_test, y_pred_test_umbral, labels=LABELS)
    metricas_any_test = metricas_any_cancer(y_test, y_pred_test_umbral)
    cm_test_umbral = confusion_matrix(y_test, y_pred_test_umbral, labels=LABELS)
    reporte_test_umbral_df = classification_report_df(y_test, y_pred_test_umbral, labels=LABELS)

    trainer.save_model(CARPETA_MODELO)
    tokenizer.save_pretrained(CARPETA_MODELO)
    guardar_json(RUTA_UMBRALES_INFERENCIA, mejores_umbrales)

    splits_resumen = [
        resumen_split("train", train_df),
        resumen_split("validation", val_df),
        resumen_split("test", test_df),
    ]

    print()
    print("Métricas de test con decisión umbralizada:")
    print(metricas_test_umbral)
    print()
    print("Métrica principal 1 o 2 vs 0:")
    print(metricas_any_test)
else:
    print("Se omitieron entrenamiento y métricas.")

Train: 2559
Validación: 640
Test: 800


Map:   0%|          | 0/2559 [00:00<?, ? examples/s]

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted,Precision Clase 0,Recall Clase 0,F1 Clase 0,Soporte Clase 0,Precision Clase 1,Recall Clase 1,F1 Clase 1,Soporte Clase 1,Precision Clase 2,Recall Clase 2,F1 Clase 2,Soporte Clase 2
1,0.046900,0.318052,0.954688,0.703710,0.847843,0.718839,0.983650,0.954687,0.967193,0.997904,0.973415,0.985507,489,0.970370,0.903448,0.935714,145,0.142857,0.666667,0.235294,6
2,0.005700,0.382095,0.973437,0.721350,0.709191,0.712119,0.972378,0.973438,0.972540,0.997921,0.981595,0.989691,489,0.916129,0.979310,0.946667,145,0.250000,0.166667,0.200000,6
3,0.014300,0.377583,0.979688,0.755161,0.711917,0.724097,0.977269,0.979687,0.978085,0.997938,0.989775,0.993840,489,0.934211,0.979310,0.956229,145,0.333333,0.166667,0.222222,6


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.



Métricas de test con decisión umbralizada:
{'accuracy': 0.94625, 'precision_macro': 0.6929956835617213, 'recall_macro': 0.8390064853179607, 'f1_macro': 0.7153778890362449, 'precision_weighted': 0.9750627909826023, 'recall_weighted': 0.9462499999999999, 'f1_weighted': 0.9587620860055752, 'precision_clase_0': 0.9965694682675815, 'recall_clase_0': 0.9524590163934427, 'f1_clase_0': 0.9740150880134116, 'soporte_clase_0': 610, 'precision_clase_1': 0.9395604395604396, 'recall_clase_1': 0.9395604395604396, 'f1_clase_1': 0.9395604395604396, 'soporte_clase_1': 182, 'precision_clase_2': 0.14285714285714285, 'recall_clase_2': 0.625, 'f1_clase_2': 0.23255813953488372, 'soporte_clase_2': 8}

Métrica principal 1 o 2 vs 0:
{'tp': 188, 'fp': 29, 'fn': 2, 'tn': 581, 'precision': 0.8663594470046083, 'recall': 0.9894736842105263, 'f1': 0.9238329238329238, 'specificity': 0.9524590163934427}


## Carga del modelo para clasificación total

In [8]:

if modo_inferencia:
    if modelo_guardado_disponible:
        tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO_INFERENCIA, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(RUTA_MODELO_INFERENCIA)
        model.to(device)
        model.eval()
        if Path(RUTA_UMBRALES_INFERENCIA).exists():
            with open(RUTA_UMBRALES_INFERENCIA, "r", encoding="utf-8") as f:
                mejores_umbrales = json.load(f)
        else:
            mejores_umbrales = {
                "t0": UMBRAL_BASE_0,
                "t1": UMBRAL_BASE_1,
                "t2": UMBRAL_BASE_2,
                "t_any": UMBRAL_BASE_ANY_CANCER,
                "margin12": MARGEN_BASE_1_SOBRE_2,
            }
    else:
        model = None
else:
    model = trainer.model
    model.to(device)
    model.eval()

print("Umbrales activos:")
print(mejores_umbrales)

Umbrales activos:
{'t0': 0.5, 't1': 0.4, 't2': 0.3, 't_any': 0.35, 'margin12': 0.05}


## Clasificación de toda la base

In [9]:

mapa_clases = {
    0: "sin_mencion_cancer",
    1: "mencion_clara_cancer",
    2: "sospecha_cancer",
}

columnas_base_resultado = []
if COLUMNA_ID in df.columns:
    columnas_base_resultado.append(COLUMNA_ID)

resultado_clases = df[columnas_base_resultado].copy().reset_index(drop=True)
resultado_clases["caso"] = df["texto_modelo"].reset_index(drop=True)

resultado_scores = df[columnas_base_resultado].copy().reset_index(drop=True)
resultado_scores["caso"] = df["texto_modelo"].reset_index(drop=True)

if puede_clasificar_total and model is not None:
    logits_all, probs_all, pred_all_argmax = predecir_dataframe_modelo(df, model, tokenizer, EVAL_BATCH_SIZE)
    pred_all_umbral, reglas_all = predecir_con_umbrales(probs_all, df["tiene_cue_sospecha"].to_numpy(dtype=int), mejores_umbrales)

    resultado_clases["clase_predicha"] = pred_all_umbral
    resultado_clases["descripcion_clase_predicha"] = resultado_clases["clase_predicha"].map(mapa_clases)

    resultado_scores["score_clase_0"] = probs_all[:, 0]
    resultado_scores["score_clase_1"] = probs_all[:, 1]
    resultado_scores["score_clase_2"] = probs_all[:, 2]
    resultado_scores["score_any_cancer"] = resultado_scores["score_clase_1"] + resultado_scores["score_clase_2"]
    resultado_scores["score_confianza_top1"] = np.max(probs_all, axis=1)
    resultado_scores["clase_top1_argmax"] = pred_all_argmax
    resultado_scores["clase_predicha_umbral"] = pred_all_umbral
    resultado_scores["regla_aplicada"] = reglas_all
else:
    resultado_clases["clase_predicha"] = pd.Series([pd.NA] * len(resultado_clases), dtype="Int64")
    resultado_clases["descripcion_clase_predicha"] = "modelo_no_disponible"

    resultado_scores["score_clase_0"] = np.nan
    resultado_scores["score_clase_1"] = np.nan
    resultado_scores["score_clase_2"] = np.nan
    resultado_scores["score_any_cancer"] = np.nan
    resultado_scores["score_confianza_top1"] = np.nan
    resultado_scores["clase_top1_argmax"] = pd.Series([pd.NA] * len(resultado_scores), dtype="Int64")
    resultado_scores["clase_predicha_umbral"] = pd.Series([pd.NA] * len(resultado_scores), dtype="Int64")
    resultado_scores["regla_aplicada"] = "modelo_no_disponible"

    print("No se generaron predicciones porque no hay un modelo afinado disponible para inferencia.")
    print("Se exportarán dos Excel con la estructura de salida para que el cuaderno no colapse con bases sin etiqueta.")

## Exportación de resultados

In [10]:
nombre_clases_excel = f"{Path(ARCHIVO_EXCEL).stem}_resultado_transformer_clases.xlsx"
ruta_clases_excel = CARPETA_SALIDA / nombre_clases_excel
resultado_clases.to_excel(ruta_clases_excel, index=False)

nombre_scores_excel = f"{Path(ARCHIVO_EXCEL).stem}_resultado_transformer_scores.xlsx"
ruta_scores_excel = CARPETA_SALIDA / nombre_scores_excel
resultado_scores.to_excel(ruta_scores_excel, index=False)

print(f"Archivo de clases guardado en: {ruta_clases_excel}")
print(f"Archivo de scores guardado en: {ruta_scores_excel}")

try:
    from google.colab import files
    files.download(str(ruta_clases_excel))
    files.download(str(ruta_scores_excel))
except Exception:
    print("Entorno local/Jupyter detectado. Los archivos quedaron en la carpeta de salida.")

Archivo de clases guardado en: resultados_roberta_hf_multiclase_umbralizado/base etiquetada limpia_resultado_transformer_clases.xlsx
Archivo de scores guardado en: resultados_roberta_hf_multiclase_umbralizado/base etiquetada limpia_resultado_transformer_scores.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
filas_metricas = []

def agregar_metricas(nombre_bloque, metricas_dict):
    if metricas_dict is None:
        return
    fila = {"bloque": nombre_bloque}
    fila.update(metricas_dict)
    filas_metricas.append(fila)

def construir_recall_por_clase(df_reporte):
    if df_reporte is None or len(df_reporte) == 0:
        return None

    df_tmp = df_reporte.copy()

    if "clase" not in df_tmp.columns:
        if "label" in df_tmp.columns:
            df_tmp = df_tmp.rename(columns={"label": "clase"})
        elif "Unnamed: 0" in df_tmp.columns:
            df_tmp = df_tmp.rename(columns={"Unnamed: 0": "clase"})
        else:
            return None

    if "recall" not in df_tmp.columns:
        if "sensibilidad" in df_tmp.columns:
            df_tmp = df_tmp.rename(columns={"sensibilidad": "recall"})
        else:
            return None

    df_tmp["clase"] = df_tmp["clase"].astype(str)
    clases_validas = {str(x) for x in LABELS}
    df_tmp = df_tmp[df_tmp["clase"].isin(clases_validas)].copy()

    if df_tmp.empty:
        return None

    df_tmp["clase"] = df_tmp["clase"].astype(int)
    df_tmp = df_tmp.sort_values("clase").reset_index(drop=True)

    return df_tmp[["clase", "recall"]]

if modo_entrenamiento:
    agregar_metricas("validacion_argmax", metricas_val_argmax)
    agregar_metricas("validacion_umbral", metricas_val_umbral)
    agregar_metricas("test_argmax", metricas_test_argmax)
    agregar_metricas("test_umbral", metricas_test_umbral)
    agregar_metricas("test_metrica_principal_any_cancer", metricas_any_test)

resumen_metricas_df = pd.DataFrame(filas_metricas)
recall_por_clase_test_df = construir_recall_por_clase(reporte_test_umbral_df)

print("Modo de ejecución:", "entrenamiento_y_evaluacion" if modo_entrenamiento else "solo_inferencia")

if modo_entrenamiento and not resumen_metricas_df.empty:
    display(resumen_metricas_df)

    if metricas_any_test is not None:
        print("\nMétrica principal en test")
        display(pd.DataFrame([metricas_any_test]))

    if recall_por_clase_test_df is not None:
        print("\nRecall por clase en test")
        display(recall_por_clase_test_df)

    if reporte_test_umbral_df is not None:
        print("\nReporte por clase en test")
        display(reporte_test_umbral_df)

    if cm_test_umbral is not None:
        print("\nMatriz de confusión en test")
        cm_df = pd.DataFrame(cm_test_umbral, index=LABELS, columns=LABELS)
        display(cm_df)
else:
    print("No se recopilaron métricas porque esta ejecución fue solo de inferencia o no hubo etiquetas válidas.")

Modo de ejecución: entrenamiento_y_evaluacion


,bloque,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,precision_clase_0,recall_clase_0,...,f1_clase_2,soporte_clase_2,tp,fp,fn,tn,precision,recall,f1,specificity
0,validacion_argmax,0.954688,0.703710,0.847843,0.718839,0.983650,0.954687,0.967193,0.997904,0.973415,...,0.235294,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,validacion_umbral,0.950000,0.699098,0.845798,0.711426,0.983510,0.950000,0.964599,0.997890,0.967280,...,0.216216,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,test_argmax,0.952500,0.700942,0.841739,0.727005,0.975323,0.952500,0.962333,0.996599,0.960656,...,0.263158,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,test_umbral,0.946250,0.692996,0.839006,0.715378,0.975063,0.946250,0.958762,0.996569,0.952459,...,0.232558,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,test_metrica_principal_any_cancer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,188.0,29.0,2.0,581.0,0.866359,0.989474,0.923833,0.952459



Métrica principal en test


,tp,fp,fn,tn,precision,recall,f1,specificity
0,188,29,2,581,0.866359,0.989474,0.923833,0.952459



Recall por clase en test


,clase,recall
0,0,0.952459
1,1,0.939560
2,2,0.625000



Reporte por clase en test


,clase,precision,sensibilidad,f1,soporte
0,0,0.996569,0.952459,0.974015,610
1,1,0.939560,0.939560,0.939560,182
2,2,0.142857,0.625000,0.232558,8



Matriz de confusión en test


,0,1,2
0,581,8,21
1,2,171,9
2,0,3,5


## Umbrales del modelo por probabilidades

### 1. Asignación de clase
La decisión final del modelo se hace con **probabilidades por clase** y **reglas de umbral**, no con scores independientes.

Sea:

- `p0`: probabilidad de clase 0 = **sin mención de cáncer**
- `p1`: probabilidad de clase 1 = **mención clara de cáncer**
- `p2`: probabilidad de clase 2 = **sospecha de cáncer**

La predicción final sigue esta lógica:

- **Clase 1** si `p1 >= t1` y `p1 >= p2 + margin12`
- **Clase 2** si `p2 >= t2`
- **Clase 2** también si hay patrón textual de sospecha y además `p1 + p2 >= t_any` con `p1 < t1`
- **Clase 0** si `p0 >= t0`
- Si ninguna regla se cumple, se usa `argmax(p0, p1, p2)` como respaldo

### 2. Umbrales base del cuaderno
Los valores base configurados en este notebook son:

- `t0 = 0.50`
- `t1 = 0.40`
- `t2 = 0.30`
- `t_any = 0.35`
- `margin12 = 0.05`

Si la base tiene suficientes etiquetas válidas, estos umbrales pueden recalibrarse automáticamente con el conjunto de validación.

### 3. Interpretación práctica
- **Clase 0**: el modelo solo niega cáncer cuando `p0` es al menos 0.50
- **Clase 1**: requiere evidencia suficiente de mención clara y una ventaja mínima frente a la clase 2
- **Clase 2**: captura sospecha explícita o casos con señal global de cáncer pero sin evidencia bastante fuerte para clase 1
- **Respaldo**: si el caso no cumple ninguna regla, se asigna la clase con mayor probabilidad

asigna los casos de 1 a 0 con confianza muy alta, superior a 95

###Informacion sobre los que fallan

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
from google.colab.data_table import DataTable

mapa_clases = {
    0: "sin_mencion_cancer",
    1: "mencion_clara_cancer",
    2: "sospecha_cancer",
}

resultado_completo = pd.DataFrame()

if 'modo_entrenamiento' in globals() and modo_entrenamiento and 'test_df' in globals():
    df_base = test_df.copy()
    print("Mostrando errores solo del conjunto de test.")
elif 'df' in globals():
    df_base = df.copy()
    print("Mostrando errores del DataFrame completo (sin modo entrenamiento o test_df no disponible).")
else:
    print("No se puede generar el informe de errores porque no hay datos base disponibles (df ni test_df).")
    df_base = pd.DataFrame()

if not df_base.empty:
    if COLUMNA_ID in df_base.columns:
        resultado_completo[COLUMNA_ID] = df_base[COLUMNA_ID].reset_index(drop=True)

    resultado_completo["caso"] = df_base["texto_modelo"].reset_index(drop=True)

    if "etiqueta_real" in df_base.columns:
        clase_real_num = pd.to_numeric(df_base["etiqueta_real"], errors="coerce")
        resultado_completo["clase_real"] = pd.Series(clase_real_num, dtype="Int64")
        resultado_completo["descripcion_clase_real"] = resultado_completo["clase_real"].map(mapa_clases)
    else:
        resultado_completo["clase_real"] = pd.Series([pd.NA] * len(df_base), dtype="Int64")
        resultado_completo["descripcion_clase_real"] = pd.NA

    if model is not None:
        logits_all, probs_all, pred_all_argmax = predecir_dataframe_modelo(
            df_base, model, tokenizer, EVAL_BATCH_SIZE
        )

        pred_all_umbral, reglas_all = predecir_con_umbrales(
            probs_all,
            df_base["tiene_cue_sospecha"].to_numpy(dtype=int),
            mejores_umbrales
        )

        resultado_completo["clase_predicha"] = pd.Series(pred_all_umbral, dtype="Int64")
        resultado_completo["descripcion_clase_predicha"] = resultado_completo["clase_predicha"].map(mapa_clases)

        resultado_completo["score_etapa1_no_cancer"] = probs_all[:, 0]
        resultado_completo["score_etapa1_any_cancer"] = probs_all[:, 1] + probs_all[:, 2]
        resultado_completo["score_etapa2_clase_1"] = probs_all[:, 1]
        resultado_completo["score_etapa2_clase_2"] = probs_all[:, 2]

        resultado_completo["score_clase_0"] = probs_all[:, 0]
        resultado_completo["score_clase_1"] = probs_all[:, 1]
        resultado_completo["score_clase_2"] = probs_all[:, 2]
        resultado_completo["score_any_cancer"] = probs_all[:, 1] + probs_all[:, 2]
        resultado_completo["score_confianza_top1"] = np.max(probs_all, axis=1)
        resultado_completo["clase_top1_argmax"] = pd.Series(pred_all_argmax, dtype="Int64")
        resultado_completo["clase_predicha_umbral"] = pd.Series(pred_all_umbral, dtype="Int64")
        resultado_completo["regla_aplicada"] = reglas_all
    else:
        resultado_completo["clase_predicha"] = pd.Series([pd.NA] * len(df_base), dtype="Int64")
        resultado_completo["descripcion_clase_predicha"] = pd.NA
        resultado_completo["score_etapa1_no_cancer"] = np.nan
        resultado_completo["score_etapa1_any_cancer"] = np.nan
        resultado_completo["score_etapa2_clase_1"] = np.nan
        resultado_completo["score_etapa2_clase_2"] = np.nan
        resultado_completo["score_clase_0"] = np.nan
        resultado_completo["score_clase_1"] = np.nan
        resultado_completo["score_clase_2"] = np.nan
        resultado_completo["score_any_cancer"] = np.nan
        resultado_completo["score_confianza_top1"] = np.nan
        resultado_completo["clase_top1_argmax"] = pd.Series([pd.NA] * len(df_base), dtype="Int64")
        resultado_completo["clase_predicha_umbral"] = pd.Series([pd.NA] * len(df_base), dtype="Int64")
        resultado_completo["regla_aplicada"] = "modelo_no_disponible"

    resultado_completo = resultado_completo[
        resultado_completo["clase_real"].notna() &
        resultado_completo["clase_predicha"].notna() &
        (resultado_completo["clase_predicha"] != resultado_completo["clase_real"])
    ].copy()

    print("Cantidad de errores encontrados:", len(resultado_completo))

    display(resultado_completo)

    display(DataTable(resultado_completo, include_index=False, num_rows_per_page=10))

    nombre_excel_completo = f"{Path(ARCHIVO_EXCEL).stem}_resultado_transformer_test_solo_errores.xlsx"
    ruta_excel_completo = CARPETA_SALIDA / nombre_excel_completo

    resultado_completo.to_excel(ruta_excel_completo, index=False)

    print(f"Archivo completo guardado en: {ruta_excel_completo}")

    try:
        from google.colab import files
        files.download(str(ruta_excel_completo))
    except Exception:
        print("Entorno local/Jupyter detectado. El archivo quedó guardado en la carpeta de salida.")

Mostrando errores solo del conjunto de test.
Cantidad de errores encontrados: 49


,ID_PSEUDO,caso,clase_real,descripcion_clase_real,clase_predicha,descripcion_clase_predicha,score_etapa1_no_cancer,score_etapa1_any_cancer,score_etapa2_clase_1,score_etapa2_clase_2,score_clase_0,score_clase_1,score_clase_2,score_any_cancer,score_confianza_top1,clase_top1_argmax,clase_predicha_umbral,regla_aplicada
1,CASO_002548,Diagnóstico A: ADENOCARCINOMA INFILTRANTE BIEN...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004520,0.995480,0.991502,0.003978,0.004520,0.991502,0.003978,0.995480,0.991502,1,1,1_por_umbral_claro
4,CASO_002963,Diagnóstico A: EMBOLIA PULMONAR [SEP] Otros es...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003410,0.996590,0.986887,0.009703,0.003410,0.986887,0.009703,0.996590,0.986887,1,1,1_por_umbral_claro
12,CASO_000815,Diagnóstico A: CÁNCER GASTRICO,0,sin_mencion_cancer,1,mencion_clara_cancer,0.005707,0.994293,0.991133,0.003161,0.005707,0.991133,0.003161,0.994293,0.991133,1,1,1_por_umbral_claro
22,CASO_003527,Diagnóstico A: PARO CARDIORESPIRATORIO [SEP] D...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003968,0.996032,0.991098,0.004934,0.003968,0.991098,0.004934,0.996032,0.991098,1,1,1_por_umbral_claro
39,CASO_003613,Diagnóstico A: PARO RESPIRATORIO [SEP] Diagnós...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004228,0.995772,0.991032,0.004740,0.004228,0.991032,0.004740,0.995772,0.991032,1,1,1_por_umbral_claro
49,CASO_003948,Diagnóstico A: FALLA VENTILATORIA [SEP] Diagnó...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003793,0.996207,0.989862,0.006346,0.003793,0.989862,0.006346,0.996207,0.989862,1,1,1_por_umbral_claro
53,CASO_002063,Diagnóstico A: SHOCK CARDIOGENICO [SEP] Diagnó...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004460,0.995540,0.991710,0.003830,0.004460,0.991710,0.003830,0.995540,0.991710,1,1,1_por_umbral_claro
93,CASO_000490,Diagnóstico A: CÁNCER BRONCOGENICO METASTÁSICO,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004673,0.995327,0.991480,0.003848,0.004673,0.991480,0.003848,0.995327,0.991480,1,1,1_por_umbral_claro
103,CASO_003642,Diagnóstico A: OBSTRUCCION INTESTINAL [SEP] Di...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003872,0.996128,0.990903,0.005225,0.003872,0.990903,0.005225,0.996128,0.990903,1,1,1_por_umbral_claro
136,CASO_002386,Diagnóstico A: DESNUTRICION PROTEICOCALORICA [...,0,sin_mencion_cancer,2,sospecha_cancer,0.665731,0.334269,0.029624,0.304645,0.665731,0.029624,0.304645,0.334269,0.665731,0,2,2_por_umbral_sospecha


,ID_PSEUDO,caso,clase_real,descripcion_clase_real,clase_predicha,descripcion_clase_predicha,score_etapa1_no_cancer,score_etapa1_any_cancer,score_etapa2_clase_1,score_etapa2_clase_2,score_clase_0,score_clase_1,score_clase_2,score_any_cancer,score_confianza_top1,clase_top1_argmax,clase_predicha_umbral,regla_aplicada
1,CASO_002548,Diagnóstico A: ADENOCARCINOMA INFILTRANTE BIEN...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004520,0.995480,0.991502,0.003978,0.004520,0.991502,0.003978,0.995480,0.991502,1,1,1_por_umbral_claro
4,CASO_002963,Diagnóstico A: EMBOLIA PULMONAR [SEP] Otros es...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003410,0.996590,0.986887,0.009703,0.003410,0.986887,0.009703,0.996590,0.986887,1,1,1_por_umbral_claro
12,CASO_000815,Diagnóstico A: CÁNCER GASTRICO,0,sin_mencion_cancer,1,mencion_clara_cancer,0.005707,0.994293,0.991133,0.003161,0.005707,0.991133,0.003161,0.994293,0.991133,1,1,1_por_umbral_claro
22,CASO_003527,Diagnóstico A: PARO CARDIORESPIRATORIO [SEP] D...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003968,0.996032,0.991098,0.004934,0.003968,0.991098,0.004934,0.996032,0.991098,1,1,1_por_umbral_claro
39,CASO_003613,Diagnóstico A: PARO RESPIRATORIO [SEP] Diagnós...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004228,0.995772,0.991032,0.004740,0.004228,0.991032,0.004740,0.995772,0.991032,1,1,1_por_umbral_claro
49,CASO_003948,Diagnóstico A: FALLA VENTILATORIA [SEP] Diagnó...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003793,0.996207,0.989862,0.006346,0.003793,0.989862,0.006346,0.996207,0.989862,1,1,1_por_umbral_claro
53,CASO_002063,Diagnóstico A: SHOCK CARDIOGENICO [SEP] Diagnó...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004460,0.995540,0.991710,0.003830,0.004460,0.991710,0.003830,0.995540,0.991710,1,1,1_por_umbral_claro
93,CASO_000490,Diagnóstico A: CÁNCER BRONCOGENICO METASTÁSICO,0,sin_mencion_cancer,1,mencion_clara_cancer,0.004673,0.995327,0.991480,0.003848,0.004673,0.991480,0.003848,0.995327,0.991480,1,1,1_por_umbral_claro
103,CASO_003642,Diagnóstico A: OBSTRUCCION INTESTINAL [SEP] Di...,0,sin_mencion_cancer,1,mencion_clara_cancer,0.003872,0.996128,0.990903,0.005225,0.003872,0.990903,0.005225,0.996128,0.990903,1,1,1_por_umbral_claro
136,CASO_002386,Diagnóstico A: DESNUTRICION PROTEICOCALORICA [...,0,sin_mencion_cancer,2,sospecha_cancer,0.665731,0.334269,0.029624,0.304645,0.665731,0.029624,0.304645,0.334269,0.665731,0,2,2_por_umbral_sospecha


Archivo completo guardado en: resultados_roberta_hf_multiclase_umbralizado/base etiquetada limpia_resultado_transformer_test_solo_errores.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import pandas as pd
from IPython.display import display

if 'splits_resumen' in globals() and splits_resumen:
    print("\nResumen de splits:")
    display(pd.DataFrame(splits_resumen))

if 'resumen_metricas_df' in globals() and not resumen_metricas_df.empty:
    print("\nMétricas de Validación y Test (Umbralizadas y Argmax):")
    display(resumen_metricas_df)

if 'metricas_any_test' in globals() and metricas_any_test is not None:
    print("\nMétrica principal de Test (Any Cancer):")
    display(pd.DataFrame([metricas_any_test]))

if 'recall_por_clase_test_df' in globals() and recall_por_clase_test_df is not None:
    print("\nRecall por Clase en Test:")
    display(recall_por_clase_test_df)

if 'reporte_test_umbral_df' in globals() and reporte_test_umbral_df is not None:
    print("\nReporte Detallado por Clase en Test (Umbralizado):")
    display(reporte_test_umbral_df)

if 'cm_test_umbral' in globals() and cm_test_umbral is not None and 'LABELS' in globals():
    print("\nMatriz de Confusión en Test (Umbralizada):")
    cm_df = pd.DataFrame(cm_test_umbral, index=LABELS, columns=LABELS)
    display(cm_df)

if 'modo_entrenamiento' not in globals():
    modo_entrenamiento = False

if not modo_entrenamiento:
    print("No se recopilaron métricas porque esta ejecución fue solo de inferencia o no hubo etiquetas válidas para el entrenamiento.")


Resumen de splits:


,nombre,n,clases
0,train,2559,"{'0': 1953, '1': 581, '2': 25}"
1,validation,640,"{'0': 489, '1': 145, '2': 6}"
2,test,800,"{'0': 610, '1': 182, '2': 8}"



Métricas de Validación y Test (Umbralizadas y Argmax):


,bloque,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,precision_clase_0,recall_clase_0,...,f1_clase_2,soporte_clase_2,tp,fp,fn,tn,precision,recall,f1,specificity
0,validacion_argmax,0.954688,0.703710,0.847843,0.718839,0.983650,0.954687,0.967193,0.997904,0.973415,...,0.235294,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,validacion_umbral,0.950000,0.699098,0.845798,0.711426,0.983510,0.950000,0.964599,0.997890,0.967280,...,0.216216,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,test_argmax,0.952500,0.700942,0.841739,0.727005,0.975323,0.952500,0.962333,0.996599,0.960656,...,0.263158,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,test_umbral,0.946250,0.692996,0.839006,0.715378,0.975063,0.946250,0.958762,0.996569,0.952459,...,0.232558,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,test_metrica_principal_any_cancer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,188.0,29.0,2.0,581.0,0.866359,0.989474,0.923833,0.952459



Métrica principal de Test (Any Cancer):


...

,clase,recall
0,0,0.952459
1,1,0.939560
2,2,0.625000



Reporte Detallado por Clase en Test (Umbralizado):


,clase,precision,sensibilidad,f1,soporte
0,0,0.996569,0.952459,0.974015,610
1,1,0.939560,0.939560,0.939560,182
2,2,0.142857,0.625000,0.232558,8



Matriz de Confusión en Test (Umbralizada):


,0,1,2
0,581,8,21
1,2,171,9
2,0,3,5
